# 1 准备以数据集

In [112]:
from transformers import AutoTokenizer


In [113]:
import os

os.environ['http_proxy'] = '127.0.0.1:10809'
os.environ['https_proxy'] = '127.0.0.1:10809'

In [114]:
tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")
tokenizer

BertTokenizer(name_or_path='hfl/rbt3', vocab_size=21128, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [115]:
tokenizer._encode_plus(["明月装饰了你的窗子", "你装饰了别人的梦"])

{'input_ids': [[101, 3209, 3299, 6163, 7652, 749, 872, 4638, 4970, 2094, 102], [101, 872, 6163, 7652, 749, 1166, 782, 4638, 3457, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [116]:
from datasets import load_dataset

In [117]:
dataset = load_dataset('lansinuote/ChnSentiCorp', cache_dir='./.cache')

In [118]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1200
    })
})

In [119]:
dataset = dataset.shuffle(seed=42)

In [120]:
from transformers import TokenizersBackend


def f(data, _tokenizer: TokenizersBackend):
    from transformers.tokenization_utils_base import TruncationStrategy
    return _tokenizer._encode_plus(data['text'], max_length=512, truncation_strategy=TruncationStrategy.LONGEST_FIRST)


dataset = dataset.map(
    function=f,
    batched=True,
    batch_size=1000,
    num_proc=4,
    remove_columns=['text'],
    fn_kwargs={"_tokenizer": tokenizer}
)
dataset

Map (num_proc=4):   0%|          | 0/9600 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1200 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1200
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1200
    })
})

In [121]:
# 删除太长的句子
def f(data):
    return [len(i) <= 512 for i in data['input_ids']]


dataset = dataset.filter(f, batched=True, batch_size=1000, num_proc=4)
dataset

Filter (num_proc=4):   0%|          | 0/9600 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/1200 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/1200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1200
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1200
    })
})

In [122]:
from transformers import AutoModelForSequenceClassification


In [123]:
model = AutoModelForSequenceClassification.from_pretrained("hfl/rbt3", num_labels=2, cache_dir="./.cache")
model

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: hfl/rbt3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your down

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

In [124]:
# 参数量
sum([i.nelement() for i in model.parameters()])

38478338

In [125]:
# 模型试算
import torch

data = {
    "input_ids": torch.ones(4, 10, dtype=torch.long),
    "token_type_ids": torch.ones(4, 10, dtype=torch.long),
    "attention_mask": torch.ones(4, 10, dtype=torch.long),
    'labels': torch.ones(4, dtype=torch.long),
}

out = model(**data)

In [126]:
out

SequenceClassifierOutput(loss=tensor(0.6307, grad_fn=<NllLossBackward0>), logits=tensor([[0.2062, 0.3353],
        [0.2062, 0.3353],
        [0.2062, 0.3353],
        [0.2062, 0.3353]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [127]:
out['logits']

tensor([[0.2062, 0.3353],
        [0.2062, 0.3353],
        [0.2062, 0.3353],
        [0.2062, 0.3353]], grad_fn=<AddmmBackward0>)

In [128]:
from evaluate import load

metric = load("accuracy", cache_dir="./.cache")

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "D:\Work\transformers-learn-demo\.venv\Lib\site-packages\huggingface_hub\utils\_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "D:\Work\transformers-learn-demo\.venv\Lib\site-packages\httpx\_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/hfl/rbt3/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "D:\Programs\Python\Python313\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "D:\Programs\Python\Python313\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwa

In [129]:
import numpy as np
from transformers.trainer_utils import EvalPrediction


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(axis=1)
    return metric.compute(predictions=pred, references=labels)


# 模拟输出
eval = EvalPrediction(
    predictions=np.array([[1, 0], [0.8, 0.2], [0.05, 0.95]]),
    label_ids=np.array([0, 1, 0]),
)
compute_metrics(eval)


{'accuracy': 0.3333333333333333}

In [130]:
# 定义训练参数
from transformers import TrainingArguments

In [148]:
train_args = TrainingArguments(
    # 定时临时数据保存路径
    output_dir="./output",
    # 定义测试执行策略，可取 no epoch steps
    eval_strategy='steps',
    # 测试间隔
    eval_steps=30,
    # 定义保存策略，可取 no epoch steps
    save_strategy='steps',
    # 保存间隔
    save_steps=30,
    # 训练轮数
    num_train_epochs=1,
    # 学习率
    learning_rate=1e-4,
    # 加入参数权重衰减 防止过拟合
    weight_decay=1e-2,
    # 定义训练和测试时候的批次大小
    per_device_eval_batch_size=16,
    per_device_train_batch_size=16,
    # 禁用报告器
    report_to=[],
    # 禁用进度条
    disable_tqdm=True,
)
train_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=True,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=30,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,


In [149]:
# 定义训练器
from transformers import Trainer
from transformers.data.data_collator import DataCollatorWithPadding

In [150]:
# 清除可能存在的旧训练器状态
import gc

gc.collect()
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[]  # 禁用所有回调，包括 NotebookProgressCallback
)
trainer

In [151]:
# 数据整理函数
temp_data = dataset['train'][:5]
[len(i) for i in temp_data['input_ids']]

[175, 136, 121, 34, 160]

In [152]:
data_collator = DataCollatorWithPadding(tokenizer)
temp_data = data_collator(temp_data)

In [153]:
# 使用padding 填充
for k, v in temp_data.items():
    print(k, v.shape)

input_ids torch.Size([5, 175])
token_type_ids torch.Size([5, 175])
attention_mask torch.Size([5, 175])
labels torch.Size([5])


In [154]:
# 可以看到 他使用了最长的句子作为最大长度 使用 PAD填充
tokenizer.decode(temp_data['input_ids'][1])

'[CLS] 感 觉 没 有 说 的 那 么 好 ！ 海 景 也 一 般 ， 给 我 安 排 的 是 很 偏 的 海 景 房 ！ 房 间 给 升 级 了 ， 不 过 是 吸 烟 房 间 ， 晚 上 回 去 发 现 问 题 严 重 ， 后 来 给 安 排 了 1 台 空 气 净 化 器 ， 问 题 算 是 解 决 了 。 房 间 很 大 ， 不 过 形 状 不 好 不 实 用 ， 到 是 挺 适 合 加 床 的 ！ 装 修 和 配 置 一 般 ， 不 能 算 豪 华 了 ！ 班 车 很 方 便 四 周 也 比 较 安 静 ， 商 业 一 般 ！ [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]'

In [155]:
# 测试和训练
# 测试
trainer.evaluate()

D:\Work\transformers-learn-demo\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.7005', 'eval_model_preparation_time': '0.001', 'eval_accuracy': '0.5083', 'eval_runtime': '35.49', 'eval_samples_per_second': '33.82', 'eval_steps_per_second': '2.114', 'epoch': 0}


{'eval_loss': 0.7004526853561401,
 'eval_model_preparation_time': 0.001,
 'eval_accuracy': 0.5083333333333333,
 'eval_runtime': 35.4859,
 'eval_samples_per_second': 33.816,
 'eval_steps_per_second': 2.114,
 'epoch': 0}

In [156]:
trainer.train()

D:\Work\transformers-learn-demo\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.4362', 'eval_model_preparation_time': '0.001', 'eval_accuracy': '0.8292', 'eval_runtime': '35.59', 'eval_samples_per_second': '33.71', 'eval_steps_per_second': '2.107', 'epoch': '0.05'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

D:\Work\transformers-learn-demo\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.3751', 'eval_model_preparation_time': '0.001', 'eval_accuracy': '0.8692', 'eval_runtime': '41.33', 'eval_samples_per_second': '29.03', 'eval_steps_per_second': '1.815', 'epoch': '0.1'}


...

D:\Work\transformers-learn-demo\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.2277', 'eval_model_preparation_time': '0.001', 'eval_accuracy': '0.9242', 'eval_runtime': '35.71', 'eval_samples_per_second': '33.6', 'eval_steps_per_second': '2.1', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '2425', 'train_samples_per_second': '3.959', 'train_steps_per_second': '0.247', 'train_loss': '0.3041', 'epoch': '1'}


TrainOutput(global_step=600, training_loss=0.3040542475382487, metrics={'train_runtime': 2425.0408, 'train_samples_per_second': 3.959, 'train_steps_per_second': 0.247, 'train_loss': 0.3040542475382487, 'epoch': 1.0})

In [157]:
# 保存和加载
import torch

In [158]:
# 默认会手动保存模型参数
# 这样可以手动保存
# trainer.save_model(output_dir="./output/save_model")
# 加载模型
# model.load_state_dict(torch.load('./output/save_model/pytorch_model.bin'))


In [ ]:
# 恢复中断的训练
# trainer.train(resume_from_checkpoint="./output/checkpoint-300")

In [ ]:
# 预测
model.eval()

for i, data in enumerate(trainer.get_eval_dataloader()):
    break
for k, v in data.items():
    data[k] = v

out = model(**data)
pred = out.logits.argmax(axis=1)